# Import libraries

In [2]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

# Get the data

In [3]:
# Load the dataset from the CSV file
dataset = pd.read_csv('../data/book700k-800k.csv')
df = pd.DataFrame({'Id': dataset['Id'],
                        'Name': dataset['Name'],
                        'Authors': dataset['Authors'],
                        'Rating': dataset['Rating'],
                        'Description': dataset['Description']})

# Display the first few rows of the dataset
df.head


<bound method NDFrame.head of            Id                                               Name  \
0      700000  A Passion to Preserve: Gay Men as Keepers of C...   
1      700002  Culture Keepers-Florida: Oral History of the A...   
2      700003  Holiday Favorites: The Best of the Williams-So...   
3      700004  Soups, Salads & Starters: the Best of Williams...   
4      700005                              Breakfasts & Brunches   
...       ...                                                ...   
54268  799991           Piano Concerto Highlights for Solo Piano   
54269  799993  Noggin King of the Nogs (The Sagas of Noggin t...   
54270  799994  No Greater Glory: The Four Immortal Chaplains ...   
54271  799996  The White Company by Arthur Conan Doyle, Ficti...   
54272  799997                    Livewire Real Lives Dawn Fraser   

                     Authors  Rating  \
0               Will Fellows    3.75   
1      Deborah Johnson-Simon    0.00   
2            Allen Rosenberg    4

In [4]:
# Replace Nan values with ''
df['Description'] = df['Description'].fillna('')
ori_description = df['Description']

In [5]:
df['Description'][0]

'From large cities to rural communities, gay men have long been impassioned pioneers as keepers of culture: rescuing and restoring decrepit buildings, revitalizing blighted neighborhoods, saving artifacts and documents of historical significance. <i>A Passion to Preserve</i> explores this authentic and complex dimension of gay men’s lives by profiling early and contemporary preservationists from throughout the United States, highlighting contributions to the larger culture that gays are exceptionally inclined to make.'

# Data preprocessing

In [6]:
import re
import nltk
nltk.download('punkt_tab')
from nltk.tokenize import word_tokenize
from nltk.corpus import stopwords
from nltk.stem import WordNetLemmatizer

nltk.download('stopwords')
nltk.download('wordnet')

# Initialize the lemmatizer
lemmatizer = WordNetLemmatizer()

# Initialize the stopwords
stop_words = set(stopwords.words('english'))

def preprocessing_text(text):
    # Convert the input text to string
    text = str(text)
    
    # Convert text to lowercase
    text = text.lower()
    
    # Remove special characters and digits and replace them with a spcae
    text = re.sub(r'[^a-zA-Z]', ' ', text)
    
    # Tokenize the text
    tokens = nltk.word_tokenize(text)
    
    # Remove stop words
    tokens = [word for word in tokens if word not in stop_words]
    
    # Lemmatize the tokens (convert words into their base dictionary form, ex. cats->cat)
    tokens = [lemmatizer.lemmatize(word, pos='v') for word in tokens]
    
    # Return the processed text as a string
    return " ".join(tokens)


def preprocess_dataframe(df, column_name):
    df[column_name ]= df[column_name].apply(preprocessing_text)
    return df

df = preprocess_dataframe(df=df, column_name='Description')


[nltk_data] Downloading package punkt_tab to
[nltk_data]     C:\Users\thlam\AppData\Roaming\nltk_data...
[nltk_data]   Package punkt_tab is already up-to-date!
[nltk_data] Downloading package stopwords to
[nltk_data]     C:\Users\thlam\AppData\Roaming\nltk_data...
[nltk_data]   Package stopwords is already up-to-date!
[nltk_data] Downloading package wordnet to
[nltk_data]     C:\Users\thlam\AppData\Roaming\nltk_data...
[nltk_data]   Package wordnet is already up-to-date!


In [7]:
df['Description'].head

<bound method NDFrame.head of 0        large cities rural communities gay men long im...
1                                                         
2        collector edition feature fabulous full color ...
3                                                         
4        america respect cookware retailer world larges...
                               ...                        
54268    concertos every pianist pinnacle performance r...
54269    king nogs br ice dragon br fly machine br omru...
54270    sink dorchester icy water greenland shortly mi...
54271    hilt cry cross narrow sea would find thick be ...
54272    book tell story dawn fraser vote greatest fema...
Name: Description, Length: 54273, dtype: str>

# Get text features using tf-idf

In [8]:
from sklearn.feature_extraction.text import TfidfVectorizer

def get_text_feature(df):
    description = df['Description']
    # Text feature
    tfidf = TfidfVectorizer(stop_words="english",
                            strip_accents='ascii',
                            token_pattern=r'\w+')

    tfidf_matrix = tfidf.fit_transform(description)
    tfidf.get_feature_names_out()
    
    return tfidf_matrix
tfidf_matrix = get_text_feature(df=df)

# Calculate cosine similarity
`from sklearn.metrics.pairwise import cosine_similarity`

In [9]:
from sklearn.metrics.pairwise import cosine_similarity

def calculate_cosine_similarity_of_a_target_book(book, tfidf_matrix):
    """Calculate the cosine similarity between a target book and all books
    in the TF-IDF matrix.

    Args:
        book (scipy.sparse.csr_matrix): 
            A single TF-IDF row vector representing the target book.
            Shape should be (1, n_features).
        tfidf_matrix (scipy.sparse.csr_matrix_): 
            TF-IDF matrix containing all book vectors.
            Shape should be (n_books, n_features).

    Returns:
        numpy.ndarray: A 2D array containing cosine similarity scores between the target book and 
        every book in the TF-IDF matrix.
        Shape will be (1, n_books).
    """
    X = book
    Y = tfidf_matrix
    cs = cosine_similarity(X, Y)
    return cs

book = calculate_cosine_similarity_of_a_target_book(tfidf_matrix=tfidf_matrix, book=tfidf_matrix[0])
book

array([[1.        , 0.        , 0.        , ..., 0.02897096, 0.        ,
        0.00840239]], shape=(1, 54273))

# Add indices to the similarity array

In [10]:
tfidf_matrix.shape[0]

54273

In [11]:
def add_indices(sim_array):
    indexed_array = []
    for i, s in zip(range(tfidf_matrix.shape[0]), sim_array[0]):
        indexed_array.append((i, s))
    return indexed_array
    
book_idx = add_indices(book)
book_idx

[(0, np.float64(1.0000000000000002)),
 (1, np.float64(0.0)),
 (2, np.float64(0.0)),
 (3, np.float64(0.0)),
 (4, np.float64(0.0)),
 (5, np.float64(0.004262619544149328)),
 (6, np.float64(0.0)),
 (7, np.float64(0.0)),
 (8, np.float64(0.02615232813989701)),
 (9, np.float64(0.005407582828292852)),
 (10, np.float64(0.006454727247923977)),
 (11, np.float64(0.001721123833796132)),
 (12, np.float64(0.033500104283204046)),
 (13, np.float64(0.0)),
 (14, np.float64(0.03093034732586701)),
 (15, np.float64(0.0)),
 (16, np.float64(0.012516813137672417)),
 (17, np.float64(0.0)),
 (18, np.float64(0.004277249851112908)),
 (19, np.float64(0.02021633193542061)),
 (20, np.float64(0.0)),
 (21, np.float64(0.0)),
 (22, np.float64(0.03540015278127994)),
 (23, np.float64(0.004704280694827859)),
 (24, np.float64(0.039896463913302024)),
 (25, np.float64(0.018351783433114693)),
 (26, np.float64(0.0)),
 (27, np.float64(0.0)),
 (28, np.float64(0.0)),
 (29, np.float64(0.0)),
 (30, np.float64(0.0)),
 (31, np.float64(

# Sort descendent

In [12]:
book_idx

[(0, np.float64(1.0000000000000002)),
 (1, np.float64(0.0)),
 (2, np.float64(0.0)),
 (3, np.float64(0.0)),
 (4, np.float64(0.0)),
 (5, np.float64(0.004262619544149328)),
 (6, np.float64(0.0)),
 (7, np.float64(0.0)),
 (8, np.float64(0.02615232813989701)),
 (9, np.float64(0.005407582828292852)),
 (10, np.float64(0.006454727247923977)),
 (11, np.float64(0.001721123833796132)),
 (12, np.float64(0.033500104283204046)),
 (13, np.float64(0.0)),
 (14, np.float64(0.03093034732586701)),
 (15, np.float64(0.0)),
 (16, np.float64(0.012516813137672417)),
 (17, np.float64(0.0)),
 (18, np.float64(0.004277249851112908)),
 (19, np.float64(0.02021633193542061)),
 (20, np.float64(0.0)),
 (21, np.float64(0.0)),
 (22, np.float64(0.03540015278127994)),
 (23, np.float64(0.004704280694827859)),
 (24, np.float64(0.039896463913302024)),
 (25, np.float64(0.018351783433114693)),
 (26, np.float64(0.0)),
 (27, np.float64(0.0)),
 (28, np.float64(0.0)),
 (29, np.float64(0.0)),
 (30, np.float64(0.0)),
 (31, np.float64(

In [13]:
def sort_des(idx_array):
    sorted_array = sorted(idx_array, key=lambda x: x[1], reverse=True)
    return sorted_array[1:11]

sorted_array = sort_des(idx_array=book_idx)
sorted_array

[(15012, np.float64(0.24589210593098956)),
 (13832, np.float64(0.23351932819036458)),
 (23708, np.float64(0.21726205019158973)),
 (17247, np.float64(0.19808039887633125)),
 (5935, np.float64(0.1928198579378127)),
 (6879, np.float64(0.1902891339739173)),
 (40495, np.float64(0.1855343589573354)),
 (16360, np.float64(0.18507251253371887)),
 (24499, np.float64(0.17768651312078693)),
 (39114, np.float64(0.1765154404979708))]

# Get the book names from the indices

In [14]:
sorted_array[1:6]

[(13832, np.float64(0.23351932819036458)),
 (23708, np.float64(0.21726205019158973)),
 (17247, np.float64(0.19808039887633125)),
 (5935, np.float64(0.1928198579378127)),
 (6879, np.float64(0.1902891339739173))]

In [15]:
def get_book_name(rec_books):
    rec_books_info = []
    for i in rec_books:
        
        rec_books_info.append(df.iloc[i[0], :])
        
    return pd.DataFrame(rec_books_info)

result = get_book_name(sorted_array)
# result = pd.DataFrame(result)
result

,Id,Name,Authors,Rating,Description
15012,727604,The Soul Beneath the Skin: The Unseen Hearts a...,David Nimmons,3.88,surprise think provoke book begin obvious fact...
13832,725500,Life Outside: The Signorile Report on Gay Men:...,Michelangelo Signorile,3.68,strong popular em em magazine columnist michel...
23708,743801,Lavender Culture,Karla Jay,3.55,influence gays lesbians language literature th...
17247,731667,Queer Wars: The New Gay Right and Its Critics,Paul A. Robinson,4.05,rebellion stonewall recent battle sex marriage...
5935,710934,Gay by the Bay: A History of Queer Culture in ...,Susan Stryker,3.89,fabulous montage word image first book ever ch...
6879,712601,Art and Sex in Greenwich Village: A Memoir of ...,Felice Picano,3.82,decade stonewall rebellions small gay press na...
40495,774575,Gay Men at the Millennium,Michael Lowenthal,4.40,core issue face gay community close millennium...
16360,730029,"If You Seduce a Straight Person, Can You Make ...",John P. De Cecco,4.00,debate whether people bear homosexual biologic...
24499,745249,Artificial Intelligence and Human Reason: A Te...,Joseph F. Rychlak,4.00,author acclaim gay fiction speak bring us new ...
39114,772020,John Gay and the London Theatre,Calhoun Winton,3.00,beggar opera often refer today first musical c...


# Put everything together

In [33]:
def get_rec_books(book_idx, df):
    selected_book = df.iloc[book_idx:book_idx+1, :]
    data = preprocess_dataframe(df=df, column_name='Description')
    matrix = get_text_feature(data)
    book_sim = calculate_cosine_similarity_of_a_target_book(book=matrix[book_idx], tfidf_matrix=matrix)
    book_sim = add_indices(book_sim)
    book_sim = sort_des(book_sim)
    result = get_book_name(rec_books=book_sim)
    
    return selected_book, result

selected_book, rec_books = get_rec_books(df=df, book_idx=1000)
    

In [34]:
rec_books

,Id,Name,Authors,Rating,Description
1,700002,Culture Keepers-Florida: Oral History of the A...,Deborah Johnson-Simon,0.00,
2,700003,Holiday Favorites: The Best of the Williams-So...,Allen Rosenberg,4.55,collector edition feature fabulous full color ...
3,700004,"Soups, Salads & Starters: the Best of Williams...",Allan Rosenberg,4.70,
4,700005,Breakfasts & Brunches,Time-Life Books,3.88,america respect cookware retailer world larges...
5,700006,Vegetarian (Best of Williams-Sonoma Kitchen Li...,Allan Rosenberg,4.18,williams sonoma almost vegetarian present reci...
6,700007,Pork and Lamb,Joanne Weir,4.00,america respect cookware retailer world larges...
7,700008,Ice Creams and Sorbets,Sarah Tenaglia,3.00,must collection kitchen america respect cookwa...
8,700011,Paradise/Tender Triumph (Omnibus),Judith McNaught,4.31,together last two acclaim novels danger desire...
9,700015,50 Thrifty Maui Restaurants: Dining on a Budge...,Yvonne Biegel,3.50,unique guidebook great food bargain price ocea...
10,700017,Frommer's Maui Day by Day,Jeanette Foster,4.14,attractively price four color guide offer doze...


In [35]:
selected_book = pd.DataFrame(selected_book)
selected_book

,Id,Name,Authors,Rating,Description
1000,701907,"Diary of a Slave Girl, Ruby Jo",K.J. McWilliams,4.26,


In [36]:
print(f"Selected book: '{selected_book['Name'].iloc[0]}'\n")
for name in rec_books['Name']: 
    print(f"{name}")

Selected book: 'Diary of a Slave Girl, Ruby Jo'

Culture Keepers-Florida: Oral History of the African American Museum Experience
Holiday Favorites: The Best of the Williams-Sonoma Kitchen Library
Soups, Salads & Starters: the Best of Williams-Sonoma Kitchen Library
Breakfasts & Brunches
Vegetarian (Best of Williams-Sonoma Kitchen Library)
Pork and Lamb
Ice Creams and Sorbets
Paradise/Tender Triumph (Omnibus)
50 Thrifty Maui Restaurants: Dining on a Budget, Island-Style
Frommer's Maui Day by Day
